# Tutorial: Evasion White-Box — FGSM e PGD

**Público:** profissionais de ML/cibersegurança com noções de PyTorch.

**Pré-requisitos:** tensores, classificação e cross-entropy.

**Objetivos:** executar FGSM/PGD com gradientes reais, medir robust accuracy e verificar o epsilon-ball.

## Roteiro

1. Preparar um classificador determinístico.
2. Gerar exemplos adversariais.
3. Comparar métricas e limites.
4. Exercício: curva de epsilon.

> Pitfall: chamar diferenças finitas sobre uma API de “white-box”. Aqui o gradiente vem de autograd.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "python"))
import torch
from torch import nn
from ml_security.whitebox_attacks_torch import TorchWhiteBoxAttacker

torch.manual_seed(42)
model = nn.Linear(2, 2)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[-10., -10.], [10., 10.]]))
    model.bias.copy_(torch.tensor([10., -10.]))
x = torch.tensor([[.20, .20], [.25, .30], [.80, .80], [.70, .75]])
y = torch.tensor([0, 0, 1, 1])
attacker = TorchWhiteBoxAttacker(0.0, 1.0, seed=42)

## Ataque e observação

FGSM usa um passo; PGD repete passos e projeta a perturbação em L∞. Ambos acessam diretamente dL/dx.

In [2]:
fgsm = attacker.fgsm(model, x, y, epsilon=0.4)
pgd = attacker.pgd(model, x, y, epsilon=0.4, alpha=0.1, steps=10)
{
    "fgsm": attacker.evaluate(model, x, y, fgsm),
    "pgd": attacker.evaluate(model, x, y, pgd),
    "pgd_max_linf": float((pgd - x).abs().max()),
}

{'fgsm': {'clean_accuracy': 1.0,
  'robust_accuracy': 0.0,
  'attack_success_rate': 1.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 1.0,
  'confidence_drop': 0.9904865026473999,
  'l2_distortion': 0.5656854510307312,
  'linf_distortion': 0.4000000059604645},
 'pgd': {'clean_accuracy': 1.0,
  'robust_accuracy': 0.0,
  'attack_success_rate': 1.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 1.0,
  'confidence_drop': 0.9904865026473999,
  'l2_distortion': 0.5656854510307312,
  'linf_distortion': 0.4000000059604645},
 'pgd_max_linf': 0.40000003576278687}

## Exercício

Varie epsilon em [0.0, 0.1, 0.2, 0.4]. A robust accuracy deve cair ou permanecer estável, nunca ser usada isoladamente como prova de robustez.

**Extensão:** troque L∞ por L2 e confira a norma por amostra.

In [3]:
curve = []
for epsilon in [0.0, 0.1, 0.2, 0.4]:
    adv = attacker.pgd(model, x, y, epsilon=epsilon, alpha=0.05, steps=10)
    curve.append({"epsilon": epsilon, **attacker.evaluate(model, x, y, adv)})
curve

[{'epsilon': 0.0,
  'clean_accuracy': 1.0,
  'robust_accuracy': 1.0,
  'attack_success_rate': 0.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 0.0,
  'confidence_drop': 0.0,
  'l2_distortion': 0.0,
  'linf_distortion': 0.0},
 {'epsilon': 0.1,
  'clean_accuracy': 1.0,
  'robust_accuracy': 1.0,
  'attack_success_rate': 0.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 0.0,
  'confidence_drop': 0.0034492015838623047,
  'l2_distortion': 0.1414213627576828,
  'linf_distortion': 0.10000000894069672},
 {'epsilon': 0.2,
  'clean_accuracy': 1.0,
  'robust_accuracy': 1.0,
  'attack_success_rate': 0.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 0.0,
  'confidence_drop': 0.14339900016784668,
  'l2_distortion': 0.2828426957130432,
  'linf_distortion': 0.19999998807907104},
 {'epsilon': 0.4,
  'clean_accuracy': 1.0,
  'robust_accuracy': 0.0,
  'attack_success_rate': 1.0,
  'targeted_success_rate': None,
  'untargeted_success_rate': 1.0,
  'confid